# Scenario Playground Step 5: Market Targets

This offline experiment constructs the 12 Scenario Playground labels from the selected Step 4 event-aware index.

Success criteria:

- use the stored pre-shock `target_baseline_date`;
- use the stored exact 21st-session `target_end_date`;
- independently validate S&P 500 rows before expanding;
- verify `DGS10` percentage-point changes become basis points;
- preserve all 585 selected rows;
- inspect every target distribution and extreme value; and
- avoid filling, shifting, clipping, splitting, features, or modeling.

In [ ]:
# Setup: locate the repository and construct targets from local caches only.
from pathlib import Path
import sys

import pandas as pd

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in (working_dir, *working_dir.parents)
    if (path / "scenario_modeling" / "src" / "targets.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.targets import build_target_artifacts

artifacts = build_target_artifacts()
targets = artifacts["targets"]
summary = artifacts["summary"]
print("Mode:", summary["construction_mode"])
print("Sampling design:", summary["sampling_design"])
print("Rows:", len(targets))
print("Targets:", len(summary["target_names"]))


## Target contract and source fields

Eleven targets are percentage returns: `100 × (future / baseline − 1)`. The 10-year target is different: `100 × (future yield percent − baseline yield percent)`, producing basis points.

Indexes, DXY, and oil futures use native `Close`. GLD and the six sector ETFs use `Adj Close` so distributions are handled consistently. The target names remain the canonical model/API names.

In [ ]:
source_table = pd.DataFrame(summary["target_specs"])[
    ["target", "source", "field", "formula", "unit"]
]
print(source_table.to_string(index=False))


## First gate: inspect S&P 500 by hand

These rows are independently recomputed with Python `Decimal` directly from the raw CSV strings. The vectorized target builder is accepted only when every independent value agrees within `1e-10` percentage points.

In [ ]:
manual = artifacts["manual_checks"]
sp500_checks = manual[manual["target"] == "sp500"]
print(sp500_checks.to_string(index=False))
assert len(sp500_checks) >= 5
assert sp500_checks["passed"].all()
print("\nS&P first-pass validation passed before all-target expansion.")


## Expansion to all 12 labels

The compact target output keeps row keys plus the 12 canonical labels. A separate audit table retains baseline and future source values so those values cannot be mistaken for model features later.

In [ ]:
print(targets.head(3).to_string(index=False))
print("\nShape:", targets.shape)
print("Missing target values:", int(targets[summary["target_names"]].isna().sum().sum()))
print("First baseline:", summary["first_target_baseline_date"])
print("Last endpoint:", summary["last_target_end_date"])
assert len(targets) == summary["input_row_count"] == 585


## Distribution review

The table shows the full 585-row distribution for every label. Large 2020 observations are retained rather than clipped because Step 5 should expose source behavior, not make a later training decision.

In [ ]:
distribution_columns = [
    "unit", "count", "mean", "standard_deviation", "minimum",
    "p01", "median", "p99", "maximum", "threshold_exceedance_count",
]
distributions = pd.DataFrame(summary["distributions"]).T[distribution_columns]
print(distributions.to_string())
assert (distributions["count"] == 585).all()


## Extreme-value review

Review thresholds are diagnostics, not filters: 50% for most returns, 150% for oil futures, and 100 basis points for the yield. Four rows cross those thresholds. They correspond to pandemic-period market behavior and remain in the dataset.

In [ ]:
extremes = artifacts["extremes"]
print(extremes.to_string(index=False))
print("\nFlagged rows:", len(extremes))
assert len(extremes) == summary["extreme_review_count"] == 4
assert (extremes["action"] == "retained_for_documented_review").all()


## Manual 10-year yield conversion

For example, the first row moves from 2.74% to 2.78%: `(2.78 − 2.74) × 100 = +4` basis points. The pandemic check moves from 1.65% to 0.54%: `−1.11 × 100 = −111` basis points. This is a yield change, not a bond-price return.

In [ ]:
yield_checks = manual[manual["target"] == "ten_year_yield_bps"]
print(yield_checks.to_string(index=False))
assert len(yield_checks) >= 3
assert yield_checks["passed"].all()


## Validation and Step 5 decision

All 585 selected rows retain their original dates. Every endpoint is exactly 21 observed `^GSPC` sessions after its baseline, and every selected source has an exact value on both dates. No target was shortened, filled, shifted, clipped, or dropped.

The target stage is ready for the next numbered step, but this notebook does not construct scenario shocks.

In [ ]:
validation = pd.Series(summary["validations"], name="passed")
print(validation.to_string())
assert validation.all()

expected_outputs = [
    "data/scenario_playground/interim/targets/market_targets.csv",
    "data/scenario_playground/interim/targets/market_target_audit.csv",
    "data/scenario_playground/interim/targets/manual_target_checks.csv",
    "data/scenario_playground/interim/targets/market_target_extremes.csv",
    "data/scenario_playground/interim/targets/market_target_summary.json",
]
assert all((PROJECT_ROOT / path).exists() for path in expected_outputs)
print("\nSTEP_5_TARGETS_READY")
